---
title: "Lab 1: NumPy - regresja liniowa"
subtitle: "Biblioteki Python w analizie danych"
author: "Tomasz Rodak"
toc-title: "Spis treści"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rodakt/BPwAD/blob/v2/laby/lab_1.ipynb)

## NumPy

Podstawowy NumPy: [Numpy: the absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html).

W zadaniach poniżej wykorzystaj:

- `rng = np.random.default_rng()` do generowania zmiennych losowych,
- wektoryzację operacji arytmetycznych, np. jeśli `A` i `B` to tablice numeryczne NumPy o takich samych wymiarach, to `A + B` zwraca tablicę, której elementy to sumy odpowiadających elementów tablic `A` i `B`,
- operator `@` do mnożenia macierzy,
- `np.linalg.inv()` do obliczenia odwrotności macierzy,
- `np.linalg.svd()` do rozkładu SVD macierzy,
- `np.transpose()` lub `.T` do transpozycji macierzy,
- `@` do mnożenia macierzy.
- `np.column_stack()` lub `np.hstack()` do łączenia tablic wzdłuż kolumn,
- `np.linalg.eigvalsh()` do obliczania wartości własnych symetrycznej macierzy rzeczywistej (wersja zoptymalizowana dla macierzy $\mathbf{X}^T\mathbf{X}$),
- `np.linalg.lstsq()` do rozwiązywania układów równań metodą najmniejszych kwadratów.

## 1. Regresja liniowa z wykorzystaniem NumPy

Celem ćwiczenia jest implementacja regresji liniowej z wykorzystaniem biblioteki NumPy. Zamiast gotowych narzędzi jak sklearn, wykorzystamy operacje algebraiczne do obliczenia współczynników modelu oraz oceny jego jakości.

Zaczniemy od wygenerowania sztucznych danych, które posłużą nam do przetestowania implementacji.


### 1.1 Generowanie danych


Niech $w_0,w_1,\ldots,w_p$, $X_1,X_2,\ldots,X_p$ oraz $\varepsilon$ będą niezależnymi zmiennymi losowymi o rozkładach:

\begin{align*}
w_i&\sim\mathcal{N}(0,1),\ i=0,1,\ldots,p,\\
X_j&\sim\mathcal{N}(0,1),\ j=1,2,\ldots,p,\\
\varepsilon&\sim\mathcal{N}(0,\sigma^2),
\end{align*}

gdzie $\mathcal{N}(\mu,\sigma^2)$ oznacza rozkład normalny o średniej $\mu$ i wariancji $\sigma^2$.

Gdy parametry $w_0,w_1,\ldots,w_p$ są znane, to zmienną zależną $Y$ definiujemy wzorem:

\begin{equation}
Y = w_0 + w_1 X_1 + w_2 X_2 + \ldots + w_p X_p + \varepsilon.
\end{equation}

Wygeneruj w Numpy zbiór danych zgodnie z powyższym modelem:

1. Wygeneruj losowy wektor współczynników $$\mathbf{w} = [w_0, w_1, \ldots, w_p]^T$$ o długości $p+1$ (uwzględniając wyraz wolny). Przyjmij $p=5$.
2. Utwórz zbiór obserwacji zmiennych niezależnych $$\mathbf{X} = [\mathbf{x}_1, \mathbf{x}_2, \ldots, \mathbf{x}_p]$$ składający się z $N=1000$ obserwacji. Symbolem $$\mathbf{x}_j=[x_{1j}, x_{2j}, \ldots, x_{Nj}]^T$$ oznaczamy wektor obserwacji zmiennej $X_j$.
3. Niech $\tilde{\mathbf{X}} = [\mathbf{1} \quad \mathbf{X}]$ będzie macierzą obserwacji $\mathbf{X}$ rozszerzoną o kolumnę jedynek (tzw. *bias term*). Wygeneruj wektor szumu $$\boldsymbol{\varepsilon} = [\varepsilon_1, \varepsilon_2, \ldots, \varepsilon_N]^T$$ o rozkładzie $\mathcal{N}(0,\sigma^2)$. Zgodnie z podanym wyżej modelem, stwórz zmienną zależną $$\mathbf{y} = \tilde{\mathbf{X}}\mathbf{w} + \boldsymbol{\varepsilon}.$$ Przyjmij $\sigma=2.0$.

### 1.2 Obliczenie całkowitej sumy kwadratów (TSS)

TSS (*Total Sum of Squares*) to suma kwadratów różnic między wartościami zmiennej zależnej `y` a ich średnią:

\begin{equation*}
\text{TSS} = \sum_{i=1}^{N} (y_i - \bar{y})^2
\end{equation*}

gdzie $\bar{y}$ to średnia wartość zmiennej `y`. Wielkość tę można interpretować jako całkowitą wariancję zmiennej zależnej. Można też powiedzieć, że jest to ocena bazowego modelu wyznaczonego przez średnią wartość zmiennej zależnej.

1. Oblicz średnią wartość zmiennej `y`.
2. Korzystając z definicji oblicz TSS.

### 1.3 Implementacja regresji liniowej

Estymator współczynników regresji $\mathbf{w}_{\text{ML}}$ wyznaczany jest za pomocą metody najmniejszych kwadratów (*Ordinary Least Squares*):

$$\mathbf{w}_{\text{ML}} = (\mathbf{\tilde X}^T\mathbf{\tilde X})^{-1}\mathbf{\tilde X}^T\mathbf{y}$$

gdzie $\mathbf{\tilde X} = [\mathbf{1} \quad \mathbf{X}]$ to macierz obserwacji $\mathbf{X}$ rozszerzona o kolumnę jedynek (tzw. *bias term*). Szczegółowe wyprowadzenie oraz właściwości statystyczne estymatora znajdziesz w artykule [Regresja liniowa w serwisie Wikipedia](https://pl.wikipedia.org/wiki/Regresja_liniowa).

1. **Oblicz współczynniki** $\mathbf{w}_{\text{ML}}$ zgodnie z powyższym wzorem macierzowym.
2. **Oblicz przewidywane wartości** zmiennej zależnej korzystając z zależności: $\hat{\mathbf{y}} = \mathbf{\tilde X} \mathbf{w}_{\text{ML}}$.
3. **Dokonaj ewaluacji modelu**: porównaj wartości estymowanych parametrów z wartościami rzeczywistymi i oblicz błędy względne dla poszczególnych obserwacji.

### 1.4 Macierz Grama i uwarunkowanie (przypadek „zdrowy”)

Niech $G = \mathbf{\tilde X}^T \mathbf{\tilde X}$. Zauważ, że $G$ jest **macierzą Grama** kolumn macierzy $\mathbf{\tilde X}$: jeśli $\mathbf{x}_0, \mathbf{x}_1,\dots,\mathbf{x}_p$ oznaczają kolumny $\mathbf{\tilde X}$, to

$$
G_{ij} = \mathbf{x}_i^T \mathbf{x}_j = \langle \mathbf{x}_i, \mathbf{x}_j\rangle,
$$

czyli $G$ zawiera wszystkie odpowiednie **iloczyny skalarne** między cechami (kolumnami).

1. Oblicz macierz $G$ i obejrzyj jej wpisy. Porównaj typową skalę elementów diagonalnych $G_{jj}$ z pozadiagonalnymi $G_{ij}$ dla $i\neq j$.
2. Oblicz wartości własne $G$ oraz liczbę uwarunkowania
   $$
   \kappa(G)=\frac{\lambda_{\max}(G)}{\lambda_{\min}(G)}.
   $$
   W tym punkcie (losowe, niezależne cechy) $\kappa(G)$ powinna być umiarkowana, a $\lambda_{\min}(G)$ nie powinna być bliska zeru.
3. Oblicz macierz odwrotną $G^{-1}$ i sprawdź, że jej wartości własne są równe $1/\lambda_i(G)$. Co to oznacza dla stabilności obliczeń przy odwracaniu macierzy?

<div class="alert alert-block alert-info">

<b>Intuicja: co „widzi” macierz Grama $G=X^T X$</b>

Macierz $G$ jest macierzą iloczynów skalarnych kolumn $X$: $G_{ij}=\langle x_i,x_j\rangle$. Elementy diagonalne $G_{jj}=\|x_j\|^2$ opisują skalę (energię) pojedynczej cechy, a wpisy pozadiagonalne $G_{ij}$ mówią, na ile dwie cechy niosą tę samą informację (korelacja / redundancja). Gdy pozadiagonalne wpisy są duże, cechy są współliniowe i trudniej „rozliczyć” ich wpływ osobno.
</div>

<div class="alert alert-block alert-warning">

<b>Dlaczego to ma znaczenie dla MNK (i czemu wrócimy do tego w sekcji 2)</b><br>
W MNK odwracamy (jawnie lub niejawnie) macierz Grama: pojawia się $G^{-1}$. Jeśli współliniowość sprawia, że najmniejsza wartość własna $\lambda_{\min}(G)$ staje się mała, to liczba uwarunkowania
$$
\kappa(G)=\frac{\lambda_{\max}(G)}{\lambda_{\min}(G)}
$$
gwałtownie rośnie, a w $G^{-1}$ pojawiają się duże wartości własne ($1/\lambda_i(G)$) i często duże wpisy. To prowadzi do niestabilnych współczynników MNK i dużej wrażliwości na szum. W sekcji 2 skonstruujemy taki przypadek celowo.
</div>

### 1.5 Ocena jakości modelu

RSS (*Residual Sum of Squares*) to suma kwadratów reszt, czyli różnic między rzeczywistymi wartościami $y_i$ a przewidywanymi $\hat{y}_i$ (jak wygląda wzór na $\hat{y}_i$?):

\begin{equation*}
\text{RSS} = \sum_{i=1}^{N} (y_i - \hat{y}_i)^2
\end{equation*}

Wielkość ta jest miarą dopasowania modelu do danych. Wyliczone w poprzednich krokach parametry modelu minimalizują RSS.

Współczynnik determinacji $R^2$ to stosunek wariancji wyjaśnionej przez model do całkowitej wariancji zmiennej zależnej:

\begin{equation*}
R^2 = \frac{\text{TSS} - \text{RSS}}{\text{TSS}}= 1 - \frac{\text{RSS}}{\text{TSS}}
\end{equation*}


1. Oblicz przewidywane wartości $\hat{y_i}$, $i=1,\ldots,N$.
2. Oblicz RSS.
3. Oblicz współczynnik determinacji $R^2$.

### Pytania dodatkowe

Zobacz jak zmienia się wartość TSS, RSS i $R^2$ w zależności od:

1. wariancji szumu,
2. liczby obserwacji $N$,
3. liczby zmiennych niezależnych $p$.

## 2. Przypadek współliniowości zmiennych

W zadaniu 1. zmienne $X_1,\ldots,X_p$ były wzajemnie niezależne, dzięki czemu macierz Grama $G = \tilde{\mathbf{X}}^T\tilde{\mathbf{X}}$ była dobrze uwarunkowana, a estymator $\mathbf{w}_{\text{ML}}$ trafnie aproksymował prawdziwe parametry modelu. Sekcja 1.4 sygnalizowała, że sytuacja zmienia się zasadniczo, gdy cechy przestają być niezależne — duże wpisy pozadiagonalne $G$ oznaczają redundancję informacji i prowadzą do niestabilności estymatora.

Teraz skonstruujemy taki przypadek celowo. Przyjmijmy, że ostatnia zmienna $X_p$ jest przybliżoną kombinacją liniową pozostałych:

$$X_p = 2X_1 - X_2 + \varepsilon_p,$$

gdzie $\varepsilon_p \sim \mathcal{N}(0, \sigma_p^2)$.

Parametr $\sigma_p$ kontroluje stopień współliniowości: dla $\sigma_p \to 0$ kolumny macierzy $\tilde{\mathbf{X}}$ stają się liniowo zależne i $G$ przestaje być odwracalna.

Przeprowadź poniższe analizy i porównaj wyniki z zadaniem 1.:

1. **Wygeneruj dane** zgodnie z powyższym modelem. Przyjmij $p=5$, $N=1000$, $\sigma=2.0$. Przetestuj kilka wartości $\sigma_p$, np. $\sigma_p \in \{0.01,\, 0.1,\, 1.0\}$.
2. **Zbadaj macierz Grama**: oblicz $G$ i jej liczbę uwarunkowania $\kappa(G)$. Jak zmienia się $\kappa(G)$ w zależności od $\sigma_p$? Porównaj z wynikiem z sekcji 1.4.
3. **Oblicz estymator $\mathbf{w}_{\text{ML}}$** metodą normalną przy użyciu `np.linalg.inv()`:
$$\mathbf{w}_{\text{ML}} = (\tilde{\mathbf{X}}^T\tilde{\mathbf{X}})^{-1}\tilde{\mathbf{X}}^T\mathbf{y}.$$
Porównaj uzyskane współczynniki z wartościami rzeczywistymi — czy estymator jest nadal trafny?
4. **Oceń jakość modelu**: oblicz RSS i $R^2$. Czy wysoki $R^2$ wyklucza niestabilność współczynników?
5. **Rozkład SVD jako alternatywa**: wyznacz $\mathbf{w}_{\text{ML}}$ za pomocą pseudoodwrotności Moore’a–Penrose’a, wykorzystując rozkład SVD zwracany przez funkcję `np.linalg.svd()`. Dla rozkładu postaci $\tilde{\mathbf{X}} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^T$ pseudoodwrotność dana jest wzorem: $$\tilde{\mathbf{X}}^+ = \mathbf{V}\boldsymbol{\Sigma}^+\mathbf{U}^T,$$ gdzie macierz $\boldsymbol{\Sigma}^+$ powstaje poprzez zastąpienie niezerowych wartości osobliwych $\sigma_i$ ich odwrotnościami $1/\sigma_i$. Estymator przyjmuje wówczas postać: $$\mathbf{w}_{\text{ML}} = \tilde{\mathbf{X}}^+\mathbf{y} = \mathbf{V}\boldsymbol{\Sigma}^+\mathbf{U}^T\mathbf{y}.$$ W celu stabilizacji numerycznej zastosuj ucięcie (*truncation*), polegające na zignorowaniu pomijalnie małych wartości osobliwych $\sigma_i$ i zastąpieniu ich zerami w macierzy $\boldsymbol{\Sigma}^+$. Porównaj stabilność wyników z metodą równań normalnych, szczególnie w przypadku złego uwarunkowania macierzy (małe $\sigma_p$).

## 3. Schemat aktywności Słońca

Na [Solar Influences Data Analysis Center (SIDC)](https://sidc.be/SILSO/datafiles) znajduje się baza danych dotycząca aktywności Słońca. Celem zadania jest wyznaczenie przeciętnej liczby plam słonecznych w zależności od czasu.

### 3.1. Wczytanie danych

Pobierz [dane (Sunspot Number)](https://sidc.be/SILSO/INFO/sndtotcsv.php) w formacie CSV. Dane te są dostępne od roku 1818. Dane możesz pobrać ręcznie, ale wygodniej będzie zrobić to wykorzystując bibliotekę `requests`:

```python
import requests

url = 'https://sidc.be/SILSO/INFO/sndtotcsv.php'
r = requests.get(url)
with open('sunspots.csv', 'wb') as f:
    f.write(r.content)
```

### 3.2. Przygotowanie i wizualizacja

1. Dane wczytaj do obiektu DataFrame biblioteki Pandas.
2. Sprawdź typy danych i usuń ewentualne braki (zobacz dokumentację zbioru na stronie SIDC).
3. Wyświetl wykres rozproszenia liczby plam słonecznych w zależności od czasu.

### 3.3 Regresja liniowa

1. Stwórz tablicę `X` zawierającą rok jako zmienną niezależną oraz tablicę `y` zawierającą liczbę plam słonecznych jako zmienną zależną.
2. Zastosuj regresję liniową do wyznaczenia przeciętnej liczby plam słonecznych w zależności od roku. Wykorzystaj bazę funkcji składających się z wyrazu wolnego oraz funkcji bazowych Gaussa: 

\begin{equation*}
\phi_i(x) = \exp\left(-\frac{(x - c_i)^2}{\sigma^2}\right),
\end{equation*}

gdzie $c_i$ to równo odległe punkty na odcinku $[X_{\text{min}},X_{\text{max}}]$ oraz $\sigma>0$ to parametr kontrolujący szerokość funkcji Gaussa.
3. Wyświetl wykres przeciętnej liczby plam słonecznych w zależności od roku wraz z wykresem rozproszenia danych.
4. Przetestuj inne typy funkcji bazowych, np. funkcję sigmoidalną 

\begin{equation*}
\phi_i(x) = \frac{1}{1 + \exp\left(-\frac{x - c_i}{\sigma}\right)}
\end{equation*}

lub ReLU

\begin{equation*}
\phi_i(x) = \max(0, x - c_i).
\end{equation*}

